# 책 추천 시스템 — 협업 필터링

- **협업 필터링** : 아이템의 내용을 몰라도, **유저들의 평점 패턴**만으로 추천하는 방식
- 앞선 실습(콘텐츠 기반)이 "책의 저자·태그"를 봤다면, 여기서는 **"누가 어떤 책에 몇 점을 줬는가"** 만 봅니다.

## 아이템 기반 최근접 이웃 협업 필터링 (Goodreads 평점 데이터)

> **"이 책을 좋아한 사람들이 좋아한 다른 책"**

각 책을 **"어떤 유저들이 몇 점을 줬는가"** 라는 벡터로 표현하고, 책끼리 코사인 유사도를 구합니다.

### 0. 데이터 준비

#### TODO 0-1. 라이브러리를 임포트하고 `books.csv`, `ratings.csv`를 불러오세요.
- `books`, `ratings` 변수에 저장하고 두 데이터의 shape을 확인

✅ **확인 포인트**: `((10000, 23), (981756, 3))`

In [1]:
# TODO 0-1: 라이브러리 임포트 + books.csv, ratings.csv 로드 → shape 확인
import numpy as np
import pandas as pd
import warnings; warnings.filterwarnings('ignore')

books = pd.read_csv('./books/books.csv')
ratings = pd.read_csv('./books/ratings.csv')

print(books.shape, ratings.shape)

(10000, 23) (981756, 3)


##### 💡 **힌트**

- `pandas`(as `pd`), `numpy`(as `np`), 경고 무시(`warnings.filterwarnings('ignore')`)
- shape은 `books.shape, ratings.shape` 처럼 튜플로 한 번에 출력할 수 있습니다.

#### TODO 0-2. 두 데이터의 상위 2개 행을 확인하세요.

In [4]:
# TODO 0-2: books.head(2), ratings.head(2) 확인
display(books.head(2).T)
display(ratings.head(2))

,0,1
id,1,2
book_id,2767052,3
best_book_id,2767052,3
work_id,2792775,4640799
books_count,272,491
isbn,439023483,439554934
isbn13,9780439023480.0,9780439554930.0
authors,Suzanne Collins,"J.K. Rowling, Mary GrandPré"
original_publication_year,2008.0,1997.0
original_title,The Hunger Games,Harry Potter and the Philosopher's Stone


,book_id,user_id,rating
0,1,314,5
1,1,439,3


##### 💡 **힌트**

- `ratings`는 `book_id`, `user_id`, `rating` 세 컬럼뿐입니다. 책 제목도 장르도 없습니다.
- `books`에는 책을 가리키는 컬럼이 **`id`와 `book_id` 두 개**입니다. 다음 단계에서 어느 쪽을 써야 하는지가 핵심입니다.

### 1. EDA, 데이터 정리

- `ratings`와 `books`를 책 번호 기준으로 merge

#### TODO 1-1. `ratings`와 `books`를 merge해 제목을 붙이세요.
- `books`에서 필요한 컬럼만 골라 `ratings`와 합치고 `rating_books`에 저장
- 상위 2개 행 확인

✅ **확인 포인트**: `(981756, 5)` — `book_id`, `user_id`, `rating`, `title`, `authors`

In [14]:
# TODO 1-1: books의 id를 book_id로 맞춰 ratings와 merge → rating_books
rating_books = pd.merge(books[['id', 'title', 'authors']], ratings, left_on='id', right_on='book_id')
rating_books.drop(['id'], axis=1, inplace=True)
rating_books.head(2)

,title,authors,book_id,user_id,rating
0,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,1,314,5
1,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,1,439,3


In [12]:
book_subs = books[['id', 'title', 'authors']].rename(columns={'id': 'book_id'})
rating_books = pd.merge(book_subs, ratings, on='book_id')
rating_books.head(2)

,book_id,title,authors,user_id,rating
0,1,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,314,5
1,1,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,439,3


##### 💡 **힌트 (중요)**

- `ratings['book_id']`는 1~10000 범위이고, **`books`의 `book_id`가 아니라 `books`의 `id`와 연결됩니다.**
  (`books['book_id']`는 Goodreads 원본 ID라 값이 훨씬 큽니다.)
- 이름이 같다고 그냥 `on='book_id'`로 합치면 **에러 없이 조용히 틀린 제목**이 붙습니다.
- 가장 깔끔한 방법: `books[['id','title','authors']]`를 뽑아 `id`를 `book_id`로 rename한 뒤 `pd.merge(ratings, b, on='book_id')`
- 검증: 책별 평균 평점과 `books['average_rating']`의 상관계수가 `id` 기준이면 **약 0.88**, `book_id` 기준이면 **약 -0.05**입니다.

#### TODO 1-2. 평점을 충분히 남긴 활성 유저만 남기세요.
- 평점 **50개 이상**인 유저만 골라 `rating_books_active`에 저장
- 남은 유저 수와 행 수 출력

✅ **확인 포인트**: 유저 **4,927명**, 행 **421,029개** (메모리 4.2GB → 약 0.4GB)

In [ ]:
# TODO 1-2: 평점 50개 이상 유저만 필터링 → rating_books_active
# user - 53424 -> 4927
# 리뷰 수 - 약 98만 개 -> 약 42만 개
# book - 10000 -> 9915
user_counts = ratings.groupby('user_id').size()
active_users = user_counts[user_counts >= 50].index
rating_books_active = rating_books[rating_books['user_id'].isin(active_users)]
# rating_books_active.groupby('book_id').size()

In [22]:
rating_books_active.head(2)

,title,authors,book_id,user_id,rating
0,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,1,314,5
1,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,1,439,3


##### 💡 **힌트**

- 유저별 개수: `rating_books.groupby('user_id').size()` → `counts[counts >= 50].index`
- 필터링: `rating_books[rating_books['user_id'].isin(대상)]`
- **왜 필요한가?** 전체 유저(53,424명) × 책(약 9,900권) 행렬은 float64로 **약 4.2GB**입니다. 그대로 `pivot_table`을 돌리면 커널이 죽습니다. 게다가 유저당 평점 중앙값이 8개뿐이라, 평점이 몇 개 없는 유저는 계산량만 늘리고 유사도에 기여하지 못합니다.

#### TODO 1-3. User-Item 행렬을 만드세요.
- `pivot_table`을 활용.
- 행: `user_id`, 열: `title`, 값: `rating`
- 결측값은 0으로 채우고 `ratings_matrix`에 저장
- shape과 상위 2개 행 확인

✅ **확인 포인트**: `(4927, 9880)`
- 열이 9,880개인 이유: 활성 유저가 평가한 책은 9,915권이고, 그중 **제목이 같은 책 35쌍이 한 열로 합쳐졌기** 때문입니다. (`The Stranger` 등 판본이 다른 동명 도서)

🤔 **생각해 볼 점**: 0으로 채운 값은 "0점"이 아니라 "안 읽음"입니다. 유사도 계산은 이 둘을 구별하지 못합니다.

In [ ]:
# TODO 1-3: pivot_table로 ratings_matrix 생성 → fillna(0) → shape, head(2) 확인
ratings_matrix = rating_books_active.pivot_table('rating', index='user_id', columns='title')
ratings_matrix.fillna(0, inplace=True)
ratings_matrix.shape

# 특정 유저에게 비슷한 유저가 읽은 책 추천 로직
# 협업 필터링으로 유저와 비슷한 유저를 찾는다 - User-Item Matrix를 Cosine Similarity 계산
# 특정 유저와 비슷한 유저들 뽑기.
# 나와 비슷한 해당 유저는 읽었는데, 특정 유저는 읽지 않은 아이템들 n개 추출

(4927, 9880)

##### 💡 **힌트**

- `df.pivot_table('rating', index='user_id', columns='title')`
- 결측 채우기: `.fillna(0, inplace=True)` 또는 `= ...fillna(0)`
- `pivot`이 아니라 `pivot_table`을 쓰는 이유: 이 데이터에는 같은 유저가 같은 책에 두 번 평점을 남긴 중복이 **2,278건** 있습니다. `pivot`은 이때 `ValueError`로 실패하지만, `pivot_table`은 평균으로 자동 집계합니다.

#### TODO 1-4. 행렬을 전치(transpose)하세요.
- `ratings_matrix_T`에 저장하고 shape과 상위 2개 행 확인

✅ **확인 포인트**: `(9880, 4927)`

In [26]:
# TODO 1-4: ratings_matrix를 전치 → ratings_matrix_T
ratings_matrix_T = ratings_matrix.T
ratings_matrix_T.shape

(9880, 4927)

##### 💡 **힌트**

- `.transpose()` 또는 `.T`
- **왜 전치하는가?** 현재는 (유저 × 책)이라 각 **행**이 유저입니다. 우리가 원하는 건 **책끼리의** 유사도이므로, 각 행이 책이 되도록 뒤집어야 합니다. 이때 한 책의 벡터는 **"4,927명의 유저가 이 책에 준 평점"** 이 됩니다.
- 콘텐츠 기반에서 책을 "단어 벡터"로 표현했다면, 여기서는 **"유저 평점 벡터"** 로 표현하는 것입니다. 유사도를 구하는 방법은 똑같습니다.

### 2. 코사인 유사도 측정

#### TODO 2-1. 책 간 코사인 유사도 행렬을 만드세요.
- `sklearn`에서 `cosine_similarity`를 임포트
- `ratings_matrix_T`로 유사도를 계산해 `item_sim`에 저장
- **제목을 인덱스와 컬럼으로 갖는** DataFrame `item_sim_df`로 만들고 shape 확인

✅ **확인 포인트**: `(9880, 9880)`, 대각 성분이 1.0

In [ ]:
# TODO 2-1: cosine_similarity로 item_sim 계산 → 제목 라벨의 item_sim_df 생성
from sklearn.metrics.pairwise import cosine_similarity

item_sim = cosine_similarity(ratings_matrix_T, ratings_matrix_T)
item_sim_df = pd.DataFrame(item_sim, 
                           index=ratings_matrix_T.index, 
                           columns=ratings_matrix_T.index)

In [30]:
item_sim_df.iloc[:3, :3]

title,"Angels (Walsh Family, #3)",#GIRLBOSS,'Salem's Lot
title,,,
"Angels (Walsh Family, #3)",1.0,0.0,0.0
#GIRLBOSS,0.0,1.0,0.0
'Salem's Lot,0.0,0.0,1.0


##### 💡 **힌트**

- 임포트: `from sklearn.metrics.pairwise import cosine_similarity`
- 자기 자신과 비교하므로 `cosine_similarity(ratings_matrix_T, ratings_matrix_T)`
- 결과는 numpy 배열이라 **제목으로 조회할 수 없습니다.** 반드시 DataFrame으로 감싸세요:
  `pd.DataFrame(data=item_sim, index=ratings_matrix.columns, columns=ratings_matrix.columns)`
  (`ratings_matrix.columns`가 곧 책 제목 목록입니다.)

### 3. 추천 책 반환 함수

#### TODO 3-1. 추천 책 DataFrame을 반환하는 함수를 작성하세요.

**함수 명세**
- 이름: `find_sim_book_item`
- 인자: `df`(유사도 DataFrame), `title_name`(기준 책 제목), `top_n`(추천 개수, 기본값 10)
- 반환: 유사도 상위 `top_n`권의 DataFrame

**동작 순서**
1. `df`에서 기준 책 컬럼만 선택
2. 자기 자신인 행을 제거
3. 유사도 내림차순 정렬 후 상위 `top_n`개 반환

In [33]:
# TODO 3-1: find_sim_book_item 함수 정의
def find_sim_book_item(df, title, top_n=10):
    sim = df[title].drop(title, axis=0)
    return sim.sort_values(ascending=False)[:top_n]

##### 💡 **힌트**

- 컬럼 하나를 **DataFrame 형태로** 선택하려면 대괄호를 두 번: `df[[title_name]]`
  (`df[title_name]`은 Series가 되어 이후 `sort_values` 사용법이 달라집니다.)
- 자기 자신 행 제거: `.drop(title_name, axis=0)` — 이렇게 하면 슬라이싱 `[1:]` 없이도 깔끔하게 빠집니다.
- 정렬: `.sort_values(title_name, ascending=False)` — DataFrame이므로 **정렬 기준 컬럼명을 넘겨야** 합니다.
- 마지막에 `[:top_n]`으로 자르기

#### TODO 3-2. 여러 책으로 추천 결과를 확인하세요.
- `'The Hobbit'`
- `'The Hunger Games (The Hunger Games, #1)'`
- `'Twilight (Twilight, #1)'`

✅ **확인 포인트**
- **The Hobbit** → 반지의 제왕 1권(0.658), 해리포터 1권(0.573), 나니아 연대기(0.534) ...
- **The Hunger Games** → 캣칭 파이어(0.595), 모킹제이(0.548) — **같은 시리즈**가 상위에 옵니다.

🎯 **주목할 점**: 이 추천은 저자도, 장르도, 태그도 **전혀 사용하지 않았습니다.** 오직 평점 패턴만으로 같은 시리즈를 찾아냈습니다.

🤔 **생각해 볼 점 — 인기도 편향(Popularity Bias)**  
결과에 `The Great Gatsby`, `To Kill a Mockingbird` 같은 **유명한 고전**이 책마다 반복해서 등장합니다. 평점을 많이 받은 책일수록 0이 아닌 값이 많아 모든 책과 유사도가 높게 나오기 때문입니다. 콘텐츠 기반 추천 결과와 비교해 보세요.

In [37]:
# TODO 3-2: 세 권에 대해 find_sim_book_item 실행
find_sim_book_item(item_sim_df, 'The Hobbit')

title
The Fellowship of the Ring (The Lord of the Rings, #1)              0.657832
Harry Potter and the Sorcerer's Stone (Harry Potter, #1)            0.572666
The Great Gatsby                                                    0.540620
The Lion, the Witch, and the Wardrobe (Chronicles of Narnia, #1)    0.533769
To Kill a Mockingbird                                               0.517456
The Adventures of Huckleberry Finn                                  0.507418
Of Mice and Men                                                     0.499250
Angels & Demons  (Robert Langdon, #1)                               0.498152
The Adventures of Tom Sawyer                                        0.496767
Lord of the Flies                                                   0.495732
Name: The Hobbit, dtype: float64

In [35]:
find_sim_book_item(item_sim_df, 'The Hunger Games (The Hunger Games, #1)')

title
Catching Fire (The Hunger Games, #2)                                0.594674
The Help                                                            0.573254
Harry Potter and the Sorcerer's Stone (Harry Potter, #1)            0.555526
Mockingjay (The Hunger Games, #3)                                   0.547697
Twilight (Twilight, #1)                                             0.509152
The Secret Garden                                                   0.488009
The Great Gatsby                                                    0.480233
The Girl with the Dragon Tattoo (Millennium, #1)                    0.477088
Angels & Demons  (Robert Langdon, #1)                               0.472808
The Lion, the Witch, and the Wardrobe (Chronicles of Narnia, #1)    0.472192
Name: The Hunger Games (The Hunger Games, #1), dtype: float64

In [36]:
find_sim_book_item(item_sim_df, 'Twilight (Twilight, #1)')

title
Harry Potter and the Sorcerer's Stone (Harry Potter, #1)    0.520216
Pride and Prejudice                                         0.513706
The Hunger Games (The Hunger Games, #1)                     0.509152
Memoirs of a Geisha                                         0.505815
The Kite Runner                                             0.488064
The Catcher in the Rye                                      0.472219
The Book Thief                                              0.470593
The Alchemist                                               0.470268
The Giver (The Giver, #1)                                   0.468721
The Diary of a Young Girl                                   0.467790
Name: Twilight (Twilight, #1), dtype: float64

---

### 정리

| 단계 | 코드 | 의미 |
|---|---|---|
| merge | `pd.merge(ratings, books[['id',...]], on='book_id')` | ⚠️ `ratings.book_id` ↔ `books.id` |
| 활성 유저 필터 | `counts[counts >= 50].index` | 전체는 4.2GB — 메모리 때문에 필수 |
| User-Item 행렬 | `pivot_table('rating', index='user_id', columns='title')` | 행=유저, 열=책, 값=평점 |
| 전치 | `.transpose()` | 책을 "유저 평점 벡터"로 표현 |
| 유사도 | `cosine_similarity(matrix_T, matrix_T)` | 책 × 책 유사도 |
| 조회 | `pd.DataFrame(sim, index=..., columns=...)` | 제목으로 바로 조회 |